# Can SymPy handle the Diophantine systems required by the logic

This is a research notebook for NTQR's development with the goal of testing whether the SymPy package can handle the large number of variables and equations involved in solving, symbolically, the linear Diophantine systems that arise whenever we want to compute the possible and consistent set of evaluations.

## First attempt

Sympy can solve a linear Diophantine system symbolically. Given the equations and a list of its variables (to distinguish them from symbolic variables for constants), it returns a symbolic solution using a new set of integer variables. This however, is not enough to construct the sets needed by the logic since these occur in a bounded region. So this exploration has two goals. The first is whether SymPy can handle systems with hundreds if not thousands of equations and variables. This could be so because only Linear Algebra is involved.

In [1]:
import itertools, random
import sympy
from IPython.display import display,Math,Latex, HTML, Javascript

import ntqr
import ntqr.raxioms

In [2]:
%%capture
sympy.init_session(quiet=True)
sympy.init_printing(print_builtin=False, use_unicode=True, use_latex='mathjax', latex_mode='plain')

def equations_to_flalign(eqs):
    lines = []
    for eq in eqs:
        lines.append(f"& {sympy.latex(eq)} &")

    body = r" \\\\ ".join(lines)

    latex_str = r"\begin{flalign*}" + body + r"\end{flalign*}"

    return latex_str

## Constructing the linear Diophantine system

We start simple by doing two label classification with three classifiers. We then check how successive solutions to the three sets look.

In [3]:
# We need to construct all the symbolic variables
# corresponding to ensemble decision events by true label
labels = ('a', 'b')
classifiers = ('i', 'j', 'k')
r_vars = [r_var for m in range(1,4)
          for m_subset in itertools.combinations(classifiers, m)
          for true_label in labels
          for r_var in ntqr.statistics.ResponseVariables(labels,m_subset).label_responses[true_label].values()]
print(len(r_vars))
random.choice(r_vars)

52


R_{b_{i},a}

In [4]:
# We expect R*(2**N - 1) = 14 simplex axioms
simplex_axioms = [axiom for m in range(1,4)
                  for m_subset in itertools.combinations(classifiers, m)
                  for axiom in ntqr.raxioms.SimplexAxioms(labels, m_subset).axioms.values()]
print(len(simplex_axioms))
random.choice(simplex_axioms)

14


-Q_b + R_{a_{j},a_{k},b} + R_{a_{j},b_{k},b} + R_{b_{j},a_{k},b} + R_{b_{j},b_ ↪

↪ {k},b}

In [5]:
# Can we solve this system symbolically
# Reverse the vars so single classifier variables are undertermined.
r_vars.reverse()

In [6]:
r_vars[0]

R_{b_{i},b_{j},b_{k},b}

In [7]:
A, b = sympy.linear_eq_to_matrix(simplex_axioms, r_vars)
A.rank()

14

In [8]:
b[0]

Qₐ

In [9]:
sol = sympy.linsolve(simplex_axioms, r_vars)

In [10]:
xs = list(list(sol)[0])
xs.reverse()
xs[0]

R_{a_{i},a}

In [11]:
xs

[R_{a_{i},a}, Qₐ - R_{a_{i},a}, R_{a_{i},b}, Q_b - R_{a_{i},b}, R_{a_{j},a}, Q ↪

↪ ₐ - R_{a_{j},a}, R_{a_{j},b}, Q_b - R_{a_{j},b}, R_{a_{k},a}, Qₐ - R_{a_{k}, ↪

↪ a}, R_{a_{k},b}, Q_b - R_{a_{k},b}, R_{a_{i},a_{j},a}, R_{a_{i},b_{j},a}, R_ ↪

↪ {b_{i},a_{j},a}, Qₐ - R_{a_{i},a_{j},a} - R_{a_{i},b_{j},a} - R_{b_{i},a_{j} ↪

↪ ,a}, R_{a_{i},a_{j},b}, R_{a_{i},b_{j},b}, R_{b_{i},a_{j},b}, Q_b - R_{a_{i} ↪

↪ ,a_{j},b} - R_{a_{i},b_{j},b} - R_{b_{i},a_{j},b}, R_{a_{i},a_{k},a}, R_{a_{ ↪

↪ i},b_{k},a}, R_{b_{i},a_{k},a}, Qₐ - R_{a_{i},a_{k},a} - R_{a_{i},b_{k},a} - ↪

↪  R_{b_{i},a_{k},a}, R_{a_{i},a_{k},b}, R_{a_{i},b_{k},b}, R_{b_{i},a_{k},b}, ↪

↪  Q_b - R_{a_{i},a_{k},b} - R_{a_{i},b_{k},b} - R_{b_{i},a_{k},b}, R_{a_{j},a ↪

↪ _{k},a}, R_{a_{j},b_{k},a}, R_{b_{j},a_{k},a}, Qₐ - R_{a_{j},a_{k},a} - R_{a ↪

↪ _{j},b_{k},a} - R_{b_{j},a_{k},a}, R_{a_{j},a_{k},b}, R_{a_{j},b_{k},b}, R_{ ↪

↪ b_{j},a_{k},b}, Q_b - R_{a_{j},a_{k},b} - R_{a_{j},b_{k},b} - R_{b_{j},a_{k} ↪

↪ ,b}, R_{a_{i},

### 2nd try
We take advantage that linsolve uses the variables at the end of its variables list as the independent ones. So we have to organize the variables by 'correct' then followed by 'error' ones.

In [12]:
correct_vars = [ntqr.statistics.ResponseVariables(labels,m_subset).correct[true_label]
          for m in range(1,4)
          for m_subset in itertools.combinations(classifiers, m)
          for true_label in labels]
print(len(correct_vars))
random.choice(correct_vars)

14


R_{a_{i},a}

In [13]:
error_vars = [error_var 
          for m in range(1,4)
          for m_subset in itertools.combinations(classifiers, m)
          for true_label in labels
          for error_var in ntqr.statistics.ResponseVariables(labels,m_subset).errors[true_label].values()]
print(len(error_vars))
random.choice(error_vars)

38


R_{b_{i},a_{j},a_{k},b}

In [14]:
all_vars = correct_vars + error_vars
A, b = sympy.linear_eq_to_matrix(simplex_axioms, all_vars)
A.rank()

14

In [15]:
len(list(sol)[0])

52

In [16]:
b

⎡Qₐ ⎤
⎢   ⎥
⎢Q_b⎥
⎢   ⎥
⎢Qₐ ⎥
⎢   ⎥
⎢Q_b⎥
⎢   ⎥
⎢Qₐ ⎥
⎢   ⎥
⎢Q_b⎥
⎢   ⎥
⎢Qₐ ⎥
⎢   ⎥
⎢Q_b⎥
⎢   ⎥
⎢Qₐ ⎥
⎢   ⎥
⎢Q_b⎥
⎢   ⎥
⎢Qₐ ⎥
⎢   ⎥
⎢Q_b⎥
⎢   ⎥
⎢Qₐ ⎥
⎢   ⎥
⎣Q_b⎦

In [17]:
list(sol)[0]

(Q_b - R_{a_{i},a_{j},a_{k},b} - R_{a_{i},a_{j},b_{k},b} - R_{a_{i},b_{j},a_{k ↪

↪ },b} - R_{a_{i},b_{j},b_{k},b} - R_{b_{i},a_{j},a_{k},b} - R_{b_{i},a_{j},b_ ↪

↪ {k},b} - R_{b_{i},b_{j},a_{k},b}, R_{b_{i},b_{j},a_{k},b}, R_{b_{i},a_{j},b_ ↪

↪ {k},b}, R_{b_{i},a_{j},a_{k},b}, R_{a_{i},b_{j},b_{k},b}, R_{a_{i},b_{j},a_{ ↪

↪ k},b}, R_{a_{i},a_{j},b_{k},b}, R_{a_{i},a_{j},a_{k},b}, Qₐ - R_{a_{i},a_{j} ↪

↪ ,a_{k},a} - R_{a_{i},a_{j},b_{k},a} - R_{a_{i},b_{j},a_{k},a} - R_{a_{i},b_{ ↪

↪ j},b_{k},a} - R_{b_{i},a_{j},a_{k},a} - R_{b_{i},a_{j},b_{k},a} - R_{b_{i},b ↪

↪ _{j},a_{k},a}, R_{b_{i},b_{j},a_{k},a}, R_{b_{i},a_{j},b_{k},a}, R_{b_{i},a_ ↪

↪ {j},a_{k},a}, R_{a_{i},b_{j},b_{k},a}, R_{a_{i},b_{j},a_{k},a}, R_{a_{i},a_{ ↪

↪ j},b_{k},a}, R_{a_{i},a_{j},a_{k},a}, Q_b - R_{a_{j},a_{k},b} - R_{a_{j},b_{ ↪

↪ k},b} - R_{b_{j},a_{k},b}, R_{b_{j},a_{k},b}, R_{a_{j},b_{k},b}, R_{a_{j},a_ ↪

↪ {k},b}, Qₐ - R_{a_{j},a_{k},a} - R_{a_{j},b_{k},a} - R_{b_{j},a_{k},a}, R_{b ↪

↪ _{j},a_{k},a},

In [18]:
all_vars

[R_{a_{i},a}, R_{b_{i},b}, R_{a_{j},a}, R_{b_{j},b}, R_{a_{k},a}, R_{b_{k},b}, ↪

↪  R_{a_{i},a_{j},a}, R_{b_{i},b_{j},b}, R_{a_{i},a_{k},a}, R_{b_{i},b_{k},b}, ↪

↪  R_{a_{j},a_{k},a}, R_{b_{j},b_{k},b}, R_{a_{i},a_{j},a_{k},a}, R_{b_{i},b_{ ↪

↪ j},b_{k},b}, R_{b_{i},a}, R_{a_{i},b}, R_{b_{j},a}, R_{a_{j},b}, R_{b_{k},a} ↪

↪ , R_{a_{k},b}, R_{a_{i},b_{j},a}, R_{b_{i},a_{j},a}, R_{b_{i},b_{j},a}, R_{a ↪

↪ _{i},a_{j},b}, R_{a_{i},b_{j},b}, R_{b_{i},a_{j},b}, R_{a_{i},b_{k},a}, R_{b ↪

↪ _{i},a_{k},a}, R_{b_{i},b_{k},a}, R_{a_{i},a_{k},b}, R_{a_{i},b_{k},b}, R_{b ↪

↪ _{i},a_{k},b}, R_{a_{j},b_{k},a}, R_{b_{j},a_{k},a}, R_{b_{j},b_{k},a}, R_{a ↪

↪ _{j},a_{k},b}, R_{a_{j},b_{k},b}, R_{b_{j},a_{k},b}, R_{a_{i},a_{j},b_{k},a} ↪

↪ , R_{a_{i},b_{j},a_{k},a}, R_{a_{i},b_{j},b_{k},a}, R_{b_{i},a_{j},a_{k},a}, ↪

↪  R_{b_{i},a_{j},b_{k},a}, R_{b_{i},b_{j},a_{k},a}, R_{b_{i},b_{j},b_{k},a},  ↪

↪ R_{a_{i},a_{j},a_{k},b}, R_{a_{i},a_{j},b_{k},b}, R_{a_{i},b_{j},a_{k},b}, R ↪

↪ _{a_{i},b_{j},

In [19]:
A

⎡1  0  0  0  0  0  0  0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0  0  0  ↪
⎢                                                                              ↪
⎢0  1  0  0  0  0  0  0  0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0  0  ↪
⎢                                                                              ↪
⎢0  0  1  0  0  0  0  0  0  0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0  ↪
⎢                                                                              ↪
⎢0  0  0  1  0  0  0  0  0  0  0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  ↪
⎢                                                                              ↪
⎢0  0  0  0  1  0  0  0  0  0  0  0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  ↪
⎢                                                                              ↪
⎢0  0  0  0  0  1  0  0  0  0  0  0  0  0  0  0  0  0  0  1  0  0  0  0  0  0  ↪
⎢                                                                              ↪
⎢0  0  0  0  0  0  1  0  0  

In [20]:
all_vars = correct_vars + error_vars
all_vars.reverse()
A, b = sympy.linear_eq_to_matrix(simplex_axioms, all_vars)
A = A.row_join(b)
sol = sympy.linsolve(A, all_vars)

In [21]:
xs=list(list(sol)[0])
xs.reverse()
xs

[R_{a_{i},a}, R_{b_{i},b}, R_{a_{j},a}, R_{b_{j},b}, R_{a_{k},a}, R_{b_{k},b}, ↪

↪  R_{a_{i},a_{j},a}, R_{b_{i},b_{j},b}, R_{a_{i},a_{k},a}, R_{b_{i},b_{k},b}, ↪

↪  R_{a_{j},a_{k},a}, R_{b_{j},b_{k},b}, R_{a_{i},a_{j},a_{k},a}, R_{b_{i},b_{ ↪

↪ j},b_{k},b}, Qₐ - R_{a_{i},a}, Q_b - R_{b_{i},b}, Qₐ - R_{a_{j},a}, Q_b - R_ ↪

↪ {b_{j},b}, Qₐ - R_{a_{k},a}, Q_b - R_{b_{k},b}, R_{a_{i},b_{j},a}, R_{b_{i}, ↪

↪ a_{j},a}, Qₐ - R_{a_{i},a_{j},a} - R_{a_{i},b_{j},a} - R_{b_{i},a_{j},a}, R_ ↪

↪ {a_{i},a_{j},b}, R_{a_{i},b_{j},b}, Q_b - R_{a_{i},a_{j},b} - R_{a_{i},b_{j} ↪

↪ ,b} - R_{b_{i},b_{j},b}, R_{a_{i},b_{k},a}, R_{b_{i},a_{k},a}, Qₐ - R_{a_{i} ↪

↪ ,a_{k},a} - R_{a_{i},b_{k},a} - R_{b_{i},a_{k},a}, R_{a_{i},a_{k},b}, R_{a_{ ↪

↪ i},b_{k},b}, Q_b - R_{a_{i},a_{k},b} - R_{a_{i},b_{k},b} - R_{b_{i},b_{k},b} ↪

↪ , R_{a_{j},b_{k},a}, R_{b_{j},a_{k},a}, Qₐ - R_{a_{j},a_{k},a} - R_{a_{j},b_ ↪

↪ {k},a} - R_{b_{j},a_{k},a}, R_{a_{j},a_{k},b}, R_{a_{j},b_{k},b}, Q_b - R_{a ↪

↪ _{j},a_{k},b} 

### Generating the possible set

The possible set does not contain any observable response variables. It is the larger set of the two. Can I figure out reasonable code to generate the set? A full evaluation is prohibitive to compute. But we can take advantage of the symmetries of the problem to organize the code. The natural variables to eliminate are the correct variables. The opposite of what I did yesterday. Why?

We expect disagreements to be less than agreements near good operating points. This is an empirical optimization since the generation is going to be carried out by iterating over the bounds of each variable. For the possible set, this is just $0 \lt R_{\ldots,\ell_t} \lt Q_{\ell_t}.$ But once we have the observables, these counts are going to be bound by $0 \lt R_{\ldots,\ell_t} \lt R_{\ldots}$ And if we get rid of the correct variables, we are iterating over smaller ranges.

Theoretically, we should expect the value of smaller subsets to be affected by observables over larger ones that subset belonged to. Can we treat these terms as we would a Taylor series? We loop over variables over smaller subsets before larger ones. How do we truncate correctly, then?

In [22]:
# Let's create the marginalization axioms
marginalization_axioms = [axiom for m in range(1,len(classifiers)+1)
                              for m_subset in itertools.combinations(classifiers,m)
                              for label in labels
                              for axiom in ntqr.raxioms.MarginalizationAxioms(labels,m_subset).axioms[label]]
len(marginalization_axioms)

48

In [23]:
all_vars = correct_vars + error_vars
#all_vars.reverse()
A, b = sympy.linear_eq_to_matrix(simplex_axioms + marginalization_axioms, all_vars)
A = A.row_join(b)
sol = sympy.linsolve(A, all_vars)
xs=list(list(sol)[0])
xs

[Qₐ - R_{b_{i},a_{j},a_{k},a} - R_{b_{i},a_{j},b_{k},a} - R_{b_{i},b_{j},a_{k} ↪

↪ ,a} - R_{b_{i},b_{j},b_{k},a}, Q_b - R_{a_{i},a_{j},a_{k},b} - R_{a_{i},a_{j ↪

↪ },b_{k},b} - R_{a_{i},b_{j},a_{k},b} - R_{a_{i},b_{j},b_{k},b}, Qₐ - R_{a_{i ↪

↪ },b_{j},a_{k},a} - R_{a_{i},b_{j},b_{k},a} - R_{b_{i},b_{j},a_{k},a} - R_{b_ ↪

↪ {i},b_{j},b_{k},a}, Q_b - R_{a_{i},a_{j},a_{k},b} - R_{a_{i},a_{j},b_{k},b}  ↪

↪ - R_{b_{i},a_{j},a_{k},b} - R_{b_{i},a_{j},b_{k},b}, Qₐ - R_{a_{i},a_{j},b_{ ↪

↪ k},a} - R_{a_{i},b_{j},b_{k},a} - R_{b_{i},a_{j},b_{k},a} - R_{b_{i},b_{j},b ↪

↪ _{k},a}, Q_b - R_{a_{i},a_{j},a_{k},b} - R_{a_{i},b_{j},a_{k},b} - R_{b_{i}, ↪

↪ a_{j},a_{k},b} - R_{b_{i},b_{j},a_{k},b}, Qₐ - R_{a_{i},b_{j},a_{k},a} - R_{ ↪

↪ a_{i},b_{j},b_{k},a} - R_{b_{i},a_{j},a_{k},a} - R_{b_{i},a_{j},b_{k},a} - R ↪

↪ _{b_{i},b_{j},a_{k},a} - R_{b_{i},b_{j},b_{k},a}, Q_b - R_{a_{i},a_{j},a_{k} ↪

↪ ,b} - R_{a_{i},a_{j},b_{k},b} - R_{a_{i},b_{j},a_{k},b} - R_{a_{i},b_{j},b_{ ↪

↪ k},b} - R_{b_{

This is simple and easy to understand. Every variable is expressed as sums over the full decision label variables. This guarantees correct marginalization. So what is the effect of the observable axioms?

In [24]:
# The observable axioms
observable_axioms = [axiom for m in range(1,len(classifiers)+1)
                              for m_subset in itertools.combinations(classifiers,m)
                              for axiom in ntqr.raxioms.ObservableAxioms(labels,m_subset).axioms]
len(observable_axioms)

26

In [25]:
# The observable axioms
observable_axioms = [axiom for axiom in ntqr.raxioms.ObservableAxioms(labels,classifiers).axioms]
len(observable_axioms)

8

In [26]:
all_vars = correct_vars + error_vars
#all_vars.reverse()
A, b = sympy.linear_eq_to_matrix(simplex_axioms + marginalization_axioms + observable_axioms, all_vars)
bvar = b[0] + b[1]
for var in b[-8:-1]:
    bvar -= var
b[-1] = bvar
A = A.row_join(b)
sol = sympy.linsolve(A, all_vars)
xs=list(list(sol)[0])
xs[0]

-R_{a_{i},a_{j},a_{k},b} + R_{a_{i},a_{j},a_{k}} - R_{a_{i},a_{j},b_{k},b} + R ↪

↪ _{a_{i},a_{j},b_{k}} - R_{a_{i},b_{j},a_{k},b} + R_{a_{i},b_{j},a_{k}} - R_{ ↪

↪ a_{i},b_{j},b_{k},b} + R_{a_{i},b_{j},b_{k}}

In [27]:
all_vars[0]

R_{a_{i},a}

In [28]:
b[-1]

Qₐ + Q_b - R_{a_{i},a_{j},a_{k}} - R_{a_{i},a_{j},b_{k}} - R_{a_{i},b_{j},a_{k ↪

↪ }} - R_{a_{i},b_{j},b_{k}} - R_{b_{i},a_{j},a_{k}} - R_{b_{i},a_{j},b_{k}} - ↪

↪  R_{b_{i},b_{j},a_{k}}

In [29]:
 xs[-1]

R_{b_{i},b_{j},a_{k},b}

In [30]:
Matrix(xs)

⎡                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                           

In [31]:
A, b = sympy.linear_eq_to_matrix(simplex_axioms + marginalization_axioms + observable_axioms, all_vars)
A.rank()

45

In [32]:
len(all_vars)

52

In [33]:
all_vars[-8]

R_{b_{i},b_{j},b_{k},a}

In [34]:
correct_vars[-1]

R_{b_{i},b_{j},b_{k},b}

In [35]:
len(correct_vars)

14

In [36]:
xs[13]

Q_b - R_{a_{i},a_{j},a_{k},b} - R_{a_{i},a_{j},b_{k},b} - R_{a_{i},b_{j},a_{k} ↪

↪ ,b} - R_{a_{i},b_{j},b_{k},b} - R_{b_{i},a_{j},a_{k},b} - R_{b_{i},a_{j},b_{ ↪

↪ k},b} - R_{b_{i},b_{j},a_{k},b}

### Doing more labels

Is this simple structure preserved as we have more labels?

In [37]:
labels=('a','b','c')
r_vars = [r_var for m in range(1,4)
          for m_subset in itertools.combinations(classifiers, m)
          for true_label in labels
          for r_var in ntqr.statistics.ResponseVariables(labels,m_subset).label_responses[true_label].values()]
print(len(r_vars))
random.choice(r_vars)

189


R_{a_{i},b_{j},b}

In [38]:
# We expect R*(2**N - 1) = 14 simplex axioms
simplex_axioms = [axiom for m in range(1,4)
                  for m_subset in itertools.combinations(classifiers, m)
                  for axiom in ntqr.raxioms.SimplexAxioms(labels, m_subset).axioms.values()]
print(len(simplex_axioms))
random.choice(simplex_axioms)

21


-Q_b + R_{a_{j},a_{k},b} + R_{a_{j},b_{k},b} + R_{a_{j},c_{k},b} + R_{b_{j},a_ ↪

↪ {k},b} + R_{b_{j},b_{k},b} + R_{b_{j},c_{k},b} + R_{c_{j},a_{k},b} + R_{c_{j ↪

↪ },b_{k},b} + R_{c_{j},c_{k},b}

In [39]:
# Let's create the marginalization axioms
marginalization_axioms = [axiom for m in range(1,len(classifiers)+1)
                              for m_subset in itertools.combinations(classifiers,m)
                              for label in labels
                              for axiom in ntqr.raxioms.MarginalizationAxioms(labels,m_subset).axioms[label]]
len(marginalization_axioms)

135

In [40]:
# The observable axioms
observable_axioms = [axiom for axiom in ntqr.raxioms.ObservableAxioms(labels,classifiers).axioms]
len(observable_axioms)

27

In [41]:
correct_vars = [ntqr.statistics.ResponseVariables(labels,m_subset).correct[true_label]
          for m in range(1,4)
          for m_subset in itertools.combinations(classifiers, m)
          for true_label in labels]
print(len(correct_vars))
random.choice(correct_vars)

21


R_{c_{i},c}

In [42]:
error_vars = [error_var 
          for m in range(1,4)
          for m_subset in itertools.combinations(classifiers, m)
          for true_label in labels
          for error_var in ntqr.statistics.ResponseVariables(labels,m_subset).errors[true_label].values()]
print(len(error_vars))
random.choice(error_vars)

168


R_{c_{i},b_{j},b_{k},b}

In [43]:
all_vars = correct_vars + error_vars
#all_vars.reverse()
A, b = sympy.linear_eq_to_matrix(simplex_axioms + marginalization_axioms + observable_axioms, all_vars)
bvar = b[0] + b[1] + b[2]
for var in b[-27:-1]:
    bvar -= var
b[-1] = bvar
A = A.row_join(b)
sol = sympy.linsolve(A, all_vars)
xs=list(list(sol)[0])
xs[0]

-R_{a_{i},a_{j},a_{k},b} - R_{a_{i},a_{j},a_{k},c} + R_{a_{i},a_{j},a_{k}} - R ↪

↪ _{a_{i},a_{j},b_{k},b} - R_{a_{i},a_{j},b_{k},c} + R_{a_{i},a_{j},b_{k}} - R ↪

↪ _{a_{i},a_{j},c_{k},b} - R_{a_{i},a_{j},c_{k},c} + R_{a_{i},a_{j},c_{k}} - R ↪

↪ _{a_{i},b_{j},a_{k},b} - R_{a_{i},b_{j},a_{k},c} + R_{a_{i},b_{j},a_{k}} - R ↪

↪ _{a_{i},b_{j},b_{k},b} - R_{a_{i},b_{j},b_{k},c} + R_{a_{i},b_{j},b_{k}} - R ↪

↪ _{a_{i},b_{j},c_{k},b} - R_{a_{i},b_{j},c_{k},c} + R_{a_{i},b_{j},c_{k}} - R ↪

↪ _{a_{i},c_{j},a_{k},b} - R_{a_{i},c_{j},a_{k},c} + R_{a_{i},c_{j},a_{k}} - R ↪

↪ _{a_{i},c_{j},b_{k},b} - R_{a_{i},c_{j},b_{k},c} + R_{a_{i},c_{j},b_{k}} - R ↪

↪ _{a_{i},c_{j},c_{k},b} - R_{a_{i},c_{j},c_{k},c} + R_{a_{i},c_{j},c_{k}}

In [44]:
bvar

Qₐ + Q_b + Q_c - R_{a_{i},a_{j},a_{k}} - R_{a_{i},a_{j},b_{k}} - R_{a_{i},a_{j ↪

↪ },c_{k}} - R_{a_{i},b_{j},a_{k}} - R_{a_{i},b_{j},b_{k}} - R_{a_{i},b_{j},c_ ↪

↪ {k}} - R_{a_{i},c_{j},a_{k}} - R_{a_{i},c_{j},b_{k}} - R_{a_{i},c_{j},c_{k}} ↪

↪  - R_{b_{i},a_{j},a_{k}} - R_{b_{i},a_{j},b_{k}} - R_{b_{i},a_{j},c_{k}} - R ↪

↪ _{b_{i},b_{j},a_{k}} - R_{b_{i},b_{j},b_{k}} - R_{b_{i},b_{j},c_{k}} - R_{b_ ↪

↪ {i},c_{j},a_{k}} - R_{b_{i},c_{j},b_{k}} - R_{b_{i},c_{j},c_{k}} - R_{c_{i}, ↪

↪ a_{j},a_{k}} - R_{c_{i},a_{j},b_{k}} - R_{c_{i},a_{j},c_{k}} - R_{c_{i},b_{j ↪

↪ },a_{k}} - R_{c_{i},b_{j},b_{k}} - R_{c_{i},b_{j},c_{k}} - R_{c_{i},c_{j},a_ ↪

↪ {k}} - R_{c_{i},c_{j},b_{k}}

In [45]:
Matrix(xs)

⎡                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                           

In [46]:
xs[-26]

R_{a_{i},a_{j},a_{k},c}

In [47]:
len(correct_vars)

21

In [48]:
all_vars[20]

R_{c_{i},c_{j},c_{k},c}

In [49]:
all_vars[19]

R_{b_{i},b_{j},b_{k},b}

In [50]:
A, b = sympy.linear_eq_to_matrix(simplex_axioms + marginalization_axioms + observable_axioms, all_vars)
A.rank()

137

In [51]:
len(all_vars)

189

In [52]:
189-137

52

In [53]:
xs[-52]

R_{a_{i},a_{j},a_{k},b}

In [54]:
xs[-53]

Qₐ + Q_b + R_{a_{i},a_{j},a_{k},c} - R_{a_{i},a_{j},a_{k}} + R_{a_{i},a_{j},b_ ↪

↪ {k},c} - R_{a_{i},a_{j},b_{k}} + R_{a_{i},a_{j},c_{k},c} - R_{a_{i},a_{j},c_ ↪

↪ {k}} + R_{a_{i},b_{j},a_{k},c} - R_{a_{i},b_{j},a_{k}} + R_{a_{i},b_{j},b_{k ↪

↪ },c} - R_{a_{i},b_{j},b_{k}} + R_{a_{i},b_{j},c_{k},c} - R_{a_{i},b_{j},c_{k ↪

↪ }} + R_{a_{i},c_{j},a_{k},c} - R_{a_{i},c_{j},a_{k}} + R_{a_{i},c_{j},b_{k}, ↪

↪ c} - R_{a_{i},c_{j},b_{k}} + R_{a_{i},c_{j},c_{k},c} - R_{a_{i},c_{j},c_{k}} ↪

↪  + R_{b_{i},a_{j},a_{k},c} - R_{b_{i},a_{j},a_{k}} + R_{b_{i},a_{j},b_{k},c} ↪

↪  - R_{b_{i},a_{j},b_{k}} + R_{b_{i},a_{j},c_{k},c} - R_{b_{i},a_{j},c_{k}} + ↪

↪  R_{b_{i},b_{j},a_{k},c} - R_{b_{i},b_{j},a_{k}} + R_{b_{i},b_{j},b_{k},c} - ↪

↪  R_{b_{i},b_{j},b_{k}} + R_{b_{i},b_{j},c_{k},c} - R_{b_{i},b_{j},c_{k}} + R ↪

↪ _{b_{i},c_{j},a_{k},c} - R_{b_{i},c_{j},a_{k}} + R_{b_{i},c_{j},b_{k},c} - R ↪

↪ _{b_{i},c_{j},b_{k}} + R_{b_{i},c_{j},c_{k},c} - R_{b_{i},c_{j},c_{k}} + R_{ ↪

↪ c_{i},a_{j},a_

In [55]:
all_vars[-54]

R_{c_{i},c_{j},b_{k},a}

In [56]:
xs[-52:]

[R_{a_{i},a_{j},a_{k},b}, R_{a_{i},a_{j},b_{k},b}, R_{a_{i},a_{j},c_{k},b}, R_ ↪

↪ {a_{i},b_{j},a_{k},b}, R_{a_{i},b_{j},b_{k},b}, R_{a_{i},b_{j},c_{k},b}, R_{ ↪

↪ a_{i},c_{j},a_{k},b}, R_{a_{i},c_{j},b_{k},b}, R_{a_{i},c_{j},c_{k},b}, R_{b ↪

↪ _{i},a_{j},a_{k},b}, R_{b_{i},a_{j},b_{k},b}, R_{b_{i},a_{j},c_{k},b}, R_{b_ ↪

↪ {i},b_{j},a_{k},b}, R_{b_{i},b_{j},c_{k},b}, R_{b_{i},c_{j},a_{k},b}, R_{b_{ ↪

↪ i},c_{j},b_{k},b}, R_{b_{i},c_{j},c_{k},b}, R_{c_{i},a_{j},a_{k},b}, R_{c_{i ↪

↪ },a_{j},b_{k},b}, R_{c_{i},a_{j},c_{k},b}, R_{c_{i},b_{j},a_{k},b}, R_{c_{i} ↪

↪ ,b_{j},b_{k},b}, R_{c_{i},b_{j},c_{k},b}, R_{c_{i},c_{j},a_{k},b}, R_{c_{i}, ↪

↪ c_{j},b_{k},b}, R_{c_{i},c_{j},c_{k},b}, R_{a_{i},a_{j},a_{k},c}, R_{a_{i},a ↪

↪ _{j},b_{k},c}, R_{a_{i},a_{j},c_{k},c}, R_{a_{i},b_{j},a_{k},c}, R_{a_{i},b_ ↪

↪ {j},b_{k},c}, R_{a_{i},b_{j},c_{k},c}, R_{a_{i},c_{j},a_{k},c}, R_{a_{i},c_{ ↪

↪ j},b_{k},c}, R_{a_{i},c_{j},c_{k},c}, R_{b_{i},a_{j},a_{k},c}, R_{b_{i},a_{j ↪

↪ },b_{k},c}, R_

In [57]:
display(xs[:21])

[-R_{a_{i},a_{j},a_{k},b} - R_{a_{i},a_{j},a_{k},c} + R_{a_{i},a_{j},a_{k}} -  ↪

↪ R_{a_{i},a_{j},b_{k},b} - R_{a_{i},a_{j},b_{k},c} + R_{a_{i},a_{j},b_{k}} -  ↪

↪ R_{a_{i},a_{j},c_{k},b} - R_{a_{i},a_{j},c_{k},c} + R_{a_{i},a_{j},c_{k}} -  ↪

↪ R_{a_{i},b_{j},a_{k},b} - R_{a_{i},b_{j},a_{k},c} + R_{a_{i},b_{j},a_{k}} -  ↪

↪ R_{a_{i},b_{j},b_{k},b} - R_{a_{i},b_{j},b_{k},c} + R_{a_{i},b_{j},b_{k}} -  ↪

↪ R_{a_{i},b_{j},c_{k},b} - R_{a_{i},b_{j},c_{k},c} + R_{a_{i},b_{j},c_{k}} -  ↪

↪ R_{a_{i},c_{j},a_{k},b} - R_{a_{i},c_{j},a_{k},c} + R_{a_{i},c_{j},a_{k}} -  ↪

↪ R_{a_{i},c_{j},b_{k},b} - R_{a_{i},c_{j},b_{k},c} + R_{a_{i},c_{j},b_{k}} -  ↪

↪ R_{a_{i},c_{j},c_{k},b} - R_{a_{i},c_{j},c_{k},c} + R_{a_{i},c_{j},c_{k}}, Q ↪

↪ _b - R_{a_{i},a_{j},a_{k},b} - R_{a_{i},a_{j},b_{k},b} - R_{a_{i},a_{j},c_{k ↪

↪ },b} - R_{a_{i},b_{j},a_{k},b} - R_{a_{i},b_{j},b_{k},b} - R_{a_{i},b_{j},c_ ↪

↪ {k},b} - R_{a_{i},c_{j},a_{k},b} - R_{a_{i},c_{j},b_{k},b} - R_{a_{i},c_{j}, ↪

↪ c_{k},b} - R_{

### Five labels

In [58]:
labels=('a','b','c','d','e')
classifiers = ('i','j','k','l')
r_vars = [r_var for m in range(1,len(classifiers)+1)
          for m_subset in itertools.combinations(classifiers, m)
          for true_label in labels
          for r_var in ntqr.statistics.ResponseVariables(labels,m_subset).label_responses[true_label].values()]
print(len(r_vars))
random.choice(r_vars)
# We expect R*(2**N - 1) = 14 simplex axioms
simplex_axioms = [axiom for m in range(1,len(classifiers)+1)
                  for m_subset in itertools.combinations(classifiers, m)
                  for axiom in ntqr.raxioms.SimplexAxioms(labels, m_subset).axioms.values()]
print(len(simplex_axioms))
random.choice(simplex_axioms)
# Let's create the marginalization axioms
marginalization_axioms = [axiom for m in range(1,len(classifiers)+1)
                              for m_subset in itertools.combinations(classifiers,m)
                              for label in labels
                              for axiom in ntqr.raxioms.MarginalizationAxioms(labels,m_subset).axioms[label]]
len(marginalization_axioms)
# The observable axioms
observable_axioms = [axiom for axiom in ntqr.raxioms.ObservableAxioms(labels,classifiers).axioms]
len(observable_axioms)
correct_vars = [ntqr.statistics.ResponseVariables(labels,m_subset).correct[true_label]
          for m in range(1,len(classifiers)+1)
          for m_subset in itertools.combinations(classifiers, m)
          for true_label in labels]
print(len(correct_vars))
random.choice(correct_vars)
error_vars = [error_var 
          for m in range(1,len(classifiers)+1)
          for m_subset in itertools.combinations(classifiers, m)
          for true_label in labels
          for error_var in ntqr.statistics.ResponseVariables(labels,m_subset).errors[true_label].values()]
print(len(error_vars))
random.choice(error_vars)
all_vars = correct_vars + error_vars
#all_vars.reverse()
A, b = sympy.linear_eq_to_matrix(simplex_axioms + marginalization_axioms + observable_axioms, all_vars)
bvar = Add(*b[:len(labels)])
nVars = (len(labels)**(len(classifiers)))
for var in b[-nVars:-1]:
    bvar -= var
b[-1] = bvar
A = A.row_join(b)
sol = sympy.linsolve(A, all_vars)
xs=list(list(sol)[0])
((len(labels)-1)*(len(labels)**len(classifiers)-1))/len(all_vars)

6475
75
75
6400


0.38548262548262546

In [59]:
labels=('a','b','c','d','e')
classifiers = ('i','j','k')
r_vars = [r_var for m in range(1,len(classifiers)+1)
          for m_subset in itertools.combinations(classifiers, m)
          for true_label in labels
          for r_var in ntqr.statistics.ResponseVariables(labels,m_subset).label_responses[true_label].values()]
print(len(r_vars))
random.choice(r_vars)
# We expect R*(2**N - 1) = 14 simplex axioms
simplex_axioms = [axiom for m in range(1,len(classifiers)+1)
                  for m_subset in itertools.combinations(classifiers, m)
                  for axiom in ntqr.raxioms.SimplexAxioms(labels, m_subset).axioms.values()]
print(len(simplex_axioms))
random.choice(simplex_axioms)
# Let's create the marginalization axioms
marginalization_axioms = [axiom for m in range(1,len(classifiers)+1)
                              for m_subset in itertools.combinations(classifiers,m)
                              for label in labels
                              for axiom in ntqr.raxioms.MarginalizationAxioms(labels,m_subset).axioms[label]]
len(marginalization_axioms)
# The observable axioms
observable_axioms = [axiom for axiom in ntqr.raxioms.ObservableAxioms(labels,classifiers).axioms]
len(observable_axioms)
correct_vars = [ntqr.statistics.ResponseVariables(labels,m_subset).correct[true_label]
          for m in range(1,len(classifiers)+1)
          for m_subset in itertools.combinations(classifiers, m)
          for true_label in labels]
print(len(correct_vars))
random.choice(correct_vars)
error_vars = [error_var 
          for m in range(1,len(classifiers)+1)
          for m_subset in itertools.combinations(classifiers, m)
          for true_label in labels
          for error_var in ntqr.statistics.ResponseVariables(labels,m_subset).errors[true_label].values()]
print(len(error_vars))
random.choice(error_vars)
all_vars = correct_vars + error_vars
#all_vars.reverse()
A, b = sympy.linear_eq_to_matrix(simplex_axioms + marginalization_axioms + observable_axioms, all_vars)
bvar = Add(*b[:len(labels)])
nVars = (len(labels)**(len(classifiers)))
for var in b[-nVars:-1]:
    bvar -= var
b[-1] = bvar
A = A.row_join(b)
sol = sympy.linsolve(A, all_vars)
xs=list(list(sol)[0])
((len(labels)-1)*(len(labels)**len(classifiers)-1))/len(all_vars)

1075
35
35
1040


0.4613953488372093

In [60]:
display(Math(equations_to_flalign([sympy.simplify(eq) for eq in xs[:15]])))

<IPython.core.display.Math object>

In [61]:
display(Math(equations_to_flalign([sympy.simplify(eq) for eq in all_vars[:15]])))

<IPython.core.display.Math object>

In [62]:
(len(labels)-1)*(len(labels)**len(classifiers)-1)

496

In [63]:
xs[-4*(5**4-1)-1]

IndexError: list index out of range

In [ ]:
4*(5**4-1)

In [ ]:
len(all_vars)

In [ ]:
2496/6475

## Major simplification

I've been letting the axioms push me around. We really have a simpler system at every Q-point. We only need to deal with the full set of classifier variables. Everything else follows as marginalization rules. This reduces the complexity of the problem.

In [ ]:
labels=('a','b','c','d','e')
classifiers = ('i','j','k')
r_vars = [r_var for true_label in labels
          for r_var in ntqr.statistics.ResponseVariables(labels,classifiers).label_responses[true_label].values()]
print(len(r_vars))
random.choice(r_vars)
# We expect R*(2**N - 1) = 14 simplex axioms
simplex_axioms = [axiom for axiom in ntqr.raxioms.SimplexAxioms(labels, classifiers).axioms.values()]
print(len(simplex_axioms))
random.choice(simplex_axioms)
# The observable axioms
observable_axioms = [axiom for axiom in ntqr.raxioms.ObservableAxioms(labels,classifiers).axioms]
len(observable_axioms)
correct_vars = [ntqr.statistics.ResponseVariables(labels,classifiers).correct[true_label]
          for true_label in labels]
print(len(correct_vars))
random.choice(correct_vars)
error_vars = [error_var for true_label in labels
          for error_var in ntqr.statistics.ResponseVariables(labels,classifiers).errors[true_label].values()]
print(len(error_vars))
random.choice(error_vars)
all_vars = correct_vars + error_vars
#all_vars.reverse()
A, b = sympy.linear_eq_to_matrix(simplex_axioms + observable_axioms, all_vars)
bvar = Add(*b[:len(labels)])
nVars = (len(labels)**(len(classifiers)))
for var in b[-nVars:-1]:
    bvar -= var
b[-1] = bvar
A = A.row_join(b)
sol = sympy.linsolve(A, all_vars)
xs=list(list(sol)[0])
((len(labels)-1)*(len(labels)**len(classifiers)-1))/len(all_vars)

In [ ]:
display(Math(equations_to_flalign(xs[:40])))

In [ ]:
all_vars[39]

In [ ]:
display(Math(equations_to_flalign(xs)))